# Untitled 2026-03-23 18:51:03



## Install dependencies

In [0]:
%pip install -r requirements.txt

## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = '5fpu8ad0qsbcox'
os.environ['DataZoneDomainId'] = 'dzd-bn7czrrcg3megx'
os.environ['DataZoneEnvironmentId'] = 'cwn4pi400pjog1'
os.environ['DataZoneDomainRegion'] = 'ap-south-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "5fpu8ad0qsbcox",
                "DataZoneDomainId": "dzd-bn7czrrcg3megx",
                "DataZoneEnvironmentId": "cwn4pi400pjog1",
                "DataZoneDomainRegion": "ap-south-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
import pandas as pd
import numpy as np
import sagemaker
import boto3
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput

In [0]:
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'sqft':      np.random.randint(800, 5000, n),
    'bedrooms':  np.random.randint(1, 6, n),
    'bathrooms': np.random.randint(1, 4, n),
    'age':       np.random.randint(1, 50, n),
})

df['price'] = (
    df['sqft'] * 150 +
    df['bedrooms'] * 10000 +
    df['bathrooms'] * 8000 -
    df['age'] * 500 +
    np.random.randint(-20000, 20000, n)
)

# price column MUST be first for SageMaker Linear Learner
df = df[['price', 'sqft', 'bedrooms', 'bathrooms', 'age']]
df.to_csv('train.csv', index=False, header=False)

print(f"Dataset shape: {df.shape}")
print(df.head())
print("train.csv created!")

Dataset shape: (200, 5)
    price  sqft  bedrooms  bathrooms  age
0  283833  1660         4          1   33
1  733450  4572         3          1   20
2  625974  3892         1          3   13
3  237754  1266         3          2   28
4  626752  4244         1          2   48
train.csv created!


In [0]:
df = pd.read_csv('train.csv', header=None)

df.dropna(inplace=True)

df.drop_duplicates(inplace=True)

# 3. Ensure all columns are numeric
df = df.apply(pd.to_numeric, errors='coerce')
df.dropna(inplace=True)

# 4. Remove outliers on price column (col 0)
Q1 = df[0].quantile(0.25)
Q3 = df[0].quantile(0.75)
IQR = Q3 - Q1
df = df[(df[0] >= Q1 - 1.5*IQR) & (df[0] <= Q3 + 1.5*IQR)]

# 5. Reset index
df.reset_index(drop=True, inplace=True)

# 6. Save cleaned file
df.to_csv('train.csv', index=False, header=False)

print(f"Shape after cleaning: {df.shape}")
print("Cleaned train.csv saved!")

Shape after cleaning: (200, 5)
Cleaned train.csv saved!


In [0]:
s3 = boto3.client('s3')

s3.upload_file(
    Filename='train.csv',
    Bucket='sagemaker-aditya006',
    Key='data/train.csv'
)

print("Upload successful!")
print("S3 path: s3://sagemaker-aditya006/data/train.csv")

Upload successful!
S3 path: s3://sagemaker-aditya006/data/train.csv


In [0]:
session = sagemaker.Session()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()

bucket = "sagemaker-aditya006"
prefix = "house-price-prediction"

print(f"Region: {region}")
print(f"Role: {role}")
print(f"Bucket: {bucket}")

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


Region: ap-south-1
Role: arn:aws:iam::833005902213:role/service-role/AmazonSageMakerAdminIAMExecutionRole
Bucket: sagemaker-aditya006


In [0]:
s3_train_path = f's3://{bucket}/data/train.csv'
container = image_uris.retrieve('linear-learner', region)

print(f"Training data: {s3_train_path}")
print(f"Container URI: {container}")

Training data: s3://sagemaker-aditya006/data/train.csv
Container URI: 991648021394.dkr.ecr.ap-south-1.amazonaws.com/linear-learner:1


In [0]:
ll = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/output',
    sagemaker_session=session
)
print("Estimator created.")

Estimator created.


In [0]:
ll.set_hyperparameters(
    feature_dim=4,
    predictor_type='regressor',
    mini_batch_size=20
)
print("Hyperparameters set.")

Hyperparameters set.


In [0]:
train_input = TrainingInput(
    s3_data=s3_train_path,
    content_type='text/csv'
)
print(f"Training input configured: {s3_train_path}")

Training input configured: s3://sagemaker-aditya006/data/train.csv


In [0]:
print("Starting training... (takes 3-5 mins)")
ll.fit({'train': train_input})
print("Training complete!")

Starting training... (takes 3-5 mins)


2026-03-23 13:38:35 Starting - Starting the training job.

.

.


2026-03-23 13:38:49 Starting - Preparing the instances for training.

...

.

Docker entrypoint called with argument(s): train
Running default environment configuration script
[03/23/2026 13:41:22 INFO 140197278799680] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss': 'auto', 'margin': '1.0', 'quantile': '0.5', 'loss_insensitivity': '0.01', 'huber_delta': '1.0', 'num_classes': '1', 'accuracy_top_k': '3', 'wd': 'auto', 'l1': 'auto', 'momentum': 'auto', 'learning_rate': 'auto', 'beta_1': 'auto', 'beta_2': 'auto', 'bias_lr_mult': 'auto', 'bias_wd_mult': 'auto', 'use_lr_scheduler': 'true', 'lr_scheduler_step': 'auto'


2026-03-23 13:41:32 Training - Training image download completed. Training in progress.
2026-03-23 13:41:32 Uploading - Uploading generated training model#metrics {"StartTime": 1774273288.2046278, "EndTime": 1774273288.2046971, "Dimensions": {"Algorithm": "Linear Learner", "Host": "algo-1", "Operation": "training", "epoch": 14, "model": 0}, "Metrics": {"train_mse_objective": {"sum": 0.18402371406555176, "count": 1, "min": 0.18402371406555176, "max": 0.18402371406555176}}}
#metrics {"StartTime": 1774273288.2048714, "EndTime": 1774273288.2048893, "Dimensions": {"Algorithm": "Linear Learner", "Host": "algo-1", "Operation": "training", "epoch": 14, "model": 1}, "Metrics": {"train_mse_objective": {"sum": 0.16965805292129515, "count": 1, "min": 0.16965805292129515, "max": 0.16965805292129515}}}
#metrics {"StartTime": 1774273288.2050354, "EndTime": 1774273288.2050512, "Dimensions": {"Algorithm": "Linear Learner", "Host": "algo-1", "Operation": "training", "epoch": 14, "model": 2}, "Metrics":


2026-03-23 13:41:51 Completed - Training job completed


Training seconds: 159
Billable seconds: 159
Training complete!


In [0]:
print("Deploying endpoint... (takes 3-5 mins)")
predictor = ll.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    serializer=sagemaker.serializers.CSVSerializer()
)
print(f"Endpoint deployed: {predictor.endpoint_name}")

Deploying endpoint... (takes 3-5 mins)


-

-

-

-

...

-

-

-

!

Endpoint deployed: linear-learner-2026-03-23-13-43-55-531


In [0]:
import boto3
import json

# Create SageMaker runtime client
runtime = boto3.client('sagemaker-runtime', region_name=region)

# Input: [sqft, bedrooms, bathrooms, age]
test_data = [[2500, 3, 2, 15]]

# Convert to CSV string
payload = '\n'.join([','.join(map(str, row)) for row in test_data])
print(f"Payload sent: {payload}")

# Call endpoint via API
response = runtime.invoke_endpoint(
    EndpointName=predictor.endpoint_name,
    ContentType='text/csv',
    Body=payload
)

# Read and parse result
result = response['Body'].read().decode('utf-8')
result_json = json.loads(result)

print("Raw response:", result)
print("Predicted price: $", result_json['predictions'][0]['score'])

Payload sent: 2500,3,2,15
Raw response: {"predictions": [{"score": 413477.375}]}
Predicted price: $ 413477.375


## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()